In [1]:
import zipfile
import shutil
import random
from pathlib import Path

import cv2
import pandas as pd

In [2]:
ZIP_PATH = Path("/content/labelled data.zip")
EXTRACT_DIR = Path("/content/data")

EXTRACT_DIR.mkdir(exist_ok=True)

print("Using zip:", ZIP_PATH)
print("Exists?", ZIP_PATH.exists())

Using zip: /content/labelled data.zip
Exists? True


In [5]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Unzip complete")

Unzip complete


In [6]:
for sub in EXTRACT_DIR.iterdir():
    print(sub)

/content/data/labelled data


In [7]:
for sub in EXTRACT_DIR.iterdir():
    if sub.is_dir():
        print(f"\n{sub.name}")
        for item in sub.iterdir():
            print("  ", item.name)


labelled data
   labelling_output_Alex3
   labelling_output_franek
   labelling_output_Alex2
   labelling_output
   labelling_output_Alex1


In [8]:
ROOT = Path("/content/data/labelled data")
MERGED_DIR = Path("/content/merged")
IMG_OUT = MERGED_DIR / "images"
CSV_OUT = MERGED_DIR / "labels.csv"

MERGED_DIR.mkdir(exist_ok=True)
IMG_OUT.mkdir(exist_ok=True)

rows = []

for folder in ROOT.iterdir():
    if not folder.is_dir():
        continue

    csv_path = folder / "liquid_labels.csv"
    img_dir = folder / "labelled_images"

    if not csv_path.exists() or not img_dir.exists():
        print(f"Skipping {folder.name} - missing liquid_labels.csv or labelled_images")
        continue

    print("Using:", folder.name)

    df = pd.read_csv(csv_path)

    for _, r in df.iterrows():
        src = img_dir / r["filename"]

        if not src.exists():
            print(f"Missing image: {src}")
            continue

        new_name = f"{folder.name}__{r['filename']}"
        dst = IMG_OUT / new_name

        shutil.copy2(src, dst)

        rows.append({
            "filename": new_name,
            "x1": r["x1"],
            "y1": r["y1"],
            "x2": r["x2"],
            "y2": r["y2"],
        })

df_all = pd.DataFrame(rows)
df_all.to_csv(CSV_OUT, index=False)

print("Total merged labelled images:", len(df_all))
print("Saved merged CSV to:", CSV_OUT)

Using: labelling_output_Alex3
Using: labelling_output_franek
Using: labelling_output_Alex2
Using: labelling_output
Using: labelling_output_Alex1
Total merged labelled images: 499
Saved merged CSV to: /content/merged/labels.csv


In [9]:
df_all.head()

,filename,x1,y1,x2,y2
0,labelling_output_Alex3__WhatsApp Image 2026-04...,734,684,894,842
1,labelling_output_Alex3__WhatsApp Image 2026-04...,502,798,640,984
2,labelling_output_Alex3__WhatsApp Image 2026-04...,410,1032,564,1246
3,labelling_output_Alex3__WhatsApp Image 2026-04...,478,712,648,918
4,labelling_output_Alex3__WhatsApp Image 2026-04...,538,770,634,882


In [10]:
DATASET = Path("/content/dataset")

for split in ["train", "val"]:
    (DATASET / f"images/{split}").mkdir(parents=True, exist_ok=True)
    (DATASET / f"labels/{split}").mkdir(parents=True, exist_ok=True)

rows = df_all.to_dict("records")
random.shuffle(rows)

split_idx = int(0.8 * len(rows))
train_rows = rows[:split_idx]
val_rows = rows[split_idx:]

print("Train images:", len(train_rows))
print("Val images:", len(val_rows))

Train images: 399
Val images: 100


In [11]:
def process_split(split_rows, split):
    for r in split_rows:
        img_path = IMG_OUT / r["filename"]

        if not img_path.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        h, w = img.shape[:2]

        x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]

        # convert to YOLO format: class x_center y_center width height
        x_center = ((x1 + x2) / 2) / w
        y_center = ((y1 + y2) / 2) / h
        box_width = (x2 - x1) / w
        box_height = (y2 - y1) / h

        # copy image
        shutil.copy2(img_path, DATASET / f"images/{split}/{r['filename']}")

        # write label file
        label_file = DATASET / f"labels/{split}/{Path(r['filename']).stem}.txt"
        with open(label_file, "w") as f:
            f.write(f"0 {x_center} {y_center} {box_width} {box_height}\n")


process_split(train_rows, "train")
process_split(val_rows, "val")

print("YOLO labels created.")

YOLO labels created.


In [12]:
yaml_text = """
path: /content/dataset
train: images/train
val: images/val

names:
  0: liquid
"""

with open("/content/dataset/data.yaml", "w") as f:
    f.write(yaml_text)

print("Created /content/dataset/data.yaml")

Created /content/dataset/data.yaml


In [13]:
for p in (DATASET / "images/train").iterdir():
    print("Example train image:", p.name)
    break

for p in (DATASET / "labels/train").iterdir():
    print("Example train label:", p.name)
    with open(p, "r") as f:
        print("Contents:", f.read())
    break

Example train image: labelling_output_Alex2__WhatsApp Image 2026-04-21 at 2.24.22 PM (16).jpeg
Example train label: labelling_output_Alex2__WhatsApp Image 2026-04-21 at 2.24.21 PM (15).txt
Contents: 0 0.5133333333333333 0.514375 0.12333333333333334 0.10125



In [14]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.4 MB/s eta 0:00:00


In [15]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [16]:
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=70,
    imgsz=640,
    batch=16,
    project="/content/runs",
    name="liquid_detector_aug",

    # geometry augmentation
    degrees=10,
    translate=0.10,
    scale=0.20,
    shear=2.0,
    perspective=0.0005,

    # colour / lighting augmentation
    hsv_h=0.05,
    hsv_s=0.7,
    hsv_v=0.5,

    # flips off for this task
    fliplr=0.0,
    flipud=0.0,

    # extra augmentation
    mosaic=0.3,
    mixup=0.0
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.05, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.3, multi_scale=0.0, name=liquid_detector_aug, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

In [20]:
from pathlib import Path

results_dir = Path("/content/runs/liquid_detector_aug")
print(results_dir.exists())
for p in results_dir.iterdir():
    print(p.name)

True
train_batch1500.jpg
weights
BoxP_curve.png
val_batch0_labels.jpg
BoxF1_curve.png
train_batch2.jpg
results.png
BoxR_curve.png
val_batch2_pred.jpg
confusion_matrix_normalized.png
val_batch1_pred.jpg
val_batch0_pred.jpg
train_batch0.jpg
confusion_matrix.png
BoxPR_curve.png
train_batch1502.jpg
results.csv
labels.jpg
val_batch1_labels.jpg
args.yaml
train_batch1.jpg
train_batch1501.jpg
val_batch2_labels.jpg


In [19]:
metrics = model.val(data="/content/dataset/data.yaml")
print(metrics)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2724.0±323.3 MB/s, size: 202.1 KB)
val: Scanning /content/dataset/labels/val.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 26.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.1it/s 3.4s
                   all        100        100      0.999          1      0.995      0.738
Speed: 4.4ms preprocess, 5.9ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bd1a03c01d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-

In [21]:
from pathlib import Path

for p in Path("/content/runs").rglob("best.pt"):
    print(p)

/content/runs/liquid_detector_aug/weights/best.pt


In [23]:
from google.colab import files
files.download("/content/runs/liquid_detector_aug/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
import shutil

shutil.make_archive(
    "/content/liquid_detector_aug_run",
    "zip",
    "/content/runs/liquid_detector_aug"
)

print("Created zip file")

Created zip file


In [25]:
from google.colab import files
files.download("/content/liquid_detector_aug_run.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>